# Create usable warp-template light curves

This notebook is the practical runtime companion to the WarpTemplate report. The construction scripts have already converted observed supernova photometry into stored phase-wavelength correction surfaces in `warpcoeff_v4/`. Here we start from those stored coefficient files and use the reusable package API to build usable `sncosmo.Model` objects and synthetic light-curve tables.

The guiding chain is:

`stored warp entry -> WarpedTimeSeriesSource -> sncosmo.Model -> band light-curve table`

The notebook deliberately keeps construction-side diagnostics separate from runtime choices. `quality`, `draw_prob`, `peakcol`, and `model_colors` describe where a stored warp came from. `z`, `t0`, `amplitude`, dust, bands, times, and zero points describe how one selected model is turned into a concrete simulated observation.


## 1. Imports and environment checks

Run this notebook from anywhere inside the workspace. The first cell finds the local `warptemplate` package and checks the modelling dependencies before importing the package-level helpers.

If a dependency is missing in this kernel, install the package into the active notebook environment, for example from the workspace root with `pip install -e ./warptemplate`.


In [ ]:
# Find the local package and check runtime dependencies
from pathlib import Path
import sys
import importlib


def find_package_dir(start=None):
    """Return the local warptemplate package directory found from start or cwd."""
    start = Path.cwd() if start is None else Path(start)

    # Walk upward so the notebook also works when opened from a nested folder.
    for path in [start, *start.parents]:
        if (path / "__init__.py").exists() and (path / "models.py").exists() and path.name == "warptemplate":
            return path
        candidate = path / "warptemplate"
        if (candidate / "__init__.py").exists() and (candidate / "models.py").exists():
            return candidate
    raise FileNotFoundError("Could not find local warptemplate package directory")


PACKAGE_DIR = find_package_dir()
WORKSPACE_DIR = PACKAGE_DIR.parent
if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))

print("PACKAGE_DIR:", PACKAGE_DIR)
print("WORKSPACE_DIR:", WORKSPACE_DIR)

required = ["numpy", "scipy", "astropy", "sncosmo", "matplotlib", "extinction", "pandas"]
missing = []
for name in required:
    try:
        mod = importlib.import_module(name)
        print(f"{name:10s}", getattr(mod, "__version__", "ok"))
    except Exception as exc:
        missing.append((name, exc))
        print(f"{name:10s} MISSING: {exc}")

HAS_RUNTIME_DEPS = len(missing) == 0
if not HAS_RUNTIME_DEPS:
    print("\nInstall the missing packages in this notebook kernel before running the modelling cells.")

In [ ]:
# Import runtime libraries when dependencies are available
if HAS_RUNTIME_DEPS:
    import pickle
    from collections import Counter

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import sncosmo
    from IPython.display import display

    from warptemplate import (
        WarpfitTemplateLoader,
        first_available_source,
        get_warpedTimeSeriesModel,
        make_identity_warpdata,
        make_lightcurve_table,
        measure_peak_color,
        plot_lightcurve_table,
    )

    print("Imported warptemplate from:", importlib.import_module("warptemplate").__file__)
else:
    print("Runtime imports skipped because dependencies are missing.")

## 2. Locate and summarize the coefficient library

The coefficient files are the runtime product of the historical construction workflow. Each file contains many observed basis SNe; each basis SN can contain several warped entries because several original `sncosmo` templates may have been acceptable starting scaffolds.

This inspection reads the pickle structure directly for orientation. The reusable runtime entry point remains `WarpfitTemplateLoader` in the next sections.


In [ ]:
# Locate coefficient files

def find_warpcoeffs_dir():
    """Return the first workspace directory that contains stored warp coefficient files."""
    candidates = [
        WORKSPACE_DIR / "warpcoeff_v4",
        WORKSPACE_DIR / "warpcoeffs",
        PACKAGE_DIR / "warpcoeffs",
    ]

    # Prefer directories that already contain versioned coefficient pickles.
    for candidate in candidates:
        if candidate.exists() and any(candidate.glob("warpcoeffs_v4_*_col.pkl")):
            return candidate
    return candidates[0]


def fitclass_from_coeff_path(path):
    """Infer the fit-class label from a v4 color-coefficient filename."""
    name = Path(path).name
    return name.removeprefix("warpcoeffs_v4_").removesuffix("_col.pkl")


def coeff_path_for_fitclass(fitclass):
    """Return the expected coefficient pickle path for a fit-class label."""
    key = fitclass.replace("/", "")
    return warpcoeffs_dir / f"warpcoeffs_v4_{key}_col.pkl"


def load_coeff_collection(fitclass):
    """Load the stored coefficient collection dictionary for one fit class."""
    with coeff_path_for_fitclass(fitclass).open("rb") as handle:
        return pickle.load(handle)


warpcoeffs_dir = find_warpcoeffs_dir()
available_coeffs = sorted(warpcoeffs_dir.glob("warpcoeffs_v4_*_col.pkl")) if warpcoeffs_dir.exists() else []

print("warpcoeffs_dir:", warpcoeffs_dir)
print("Found", len(available_coeffs), "coefficient files")
for coeff in available_coeffs[:20]:
    print(" -", coeff.name)

if not available_coeffs:
    print("No coefficient files found. Set warpcoeffs_dir manually before running the sampling cells.")

In [ ]:
# Summarize coefficient-library contents

def summarize_coeff_file(path):
    """Return one summary row for a stored warp-coefficient pickle file."""
    fitclass = fitclass_from_coeff_path(path)
    with Path(path).open("rb") as handle:
        collection = pickle.load(handle)

    # Flatten the nested basis-SN -> stored-entry structure for quick counting.
    warpcoeff = collection.get("warpcoeff", {})
    entries = [entry for basis_entries in warpcoeff.values() for entry in basis_entries]
    qualities = Counter(entry.get("quality", "missing") for entry in entries)
    templates = {entry.get("model") for entry in entries}
    peakcols = [entry.get("peakcol") for entry in entries if entry.get("peakcol") is not None]
    model_colors = collection.get("model_colors") or {}

    return {
        "fitclass": fitclass,
        "basis_sne": len(warpcoeff),
        "entries": len(entries),
        "templates": len(templates),
        "gold": qualities.get("gold", 0),
        "silver": qualities.get("silver", 0),
        "bronze": qualities.get("bronze", 0),
        "color_pair": "-".join([model_colors.get("color1", ""), model_colors.get("color2", "")]).strip("-"),
        "peakcol_count": len(peakcols),
        "peakcol_median": float(np.median(peakcols)) if peakcols else np.nan,
    }


if HAS_RUNTIME_DEPS and available_coeffs:
    library_summary = pd.DataFrame([summarize_coeff_file(path) for path in available_coeffs])
    library_summary = library_summary.sort_values(["entries", "basis_sne"], ascending=False).reset_index(drop=True)
    display(library_summary)

    totals = library_summary[["basis_sne", "entries", "templates", "gold", "silver", "bronze"]].sum()
    print("Total basis SNe:", int(totals["basis_sne"]))
    print("Total stored warp entries:", int(totals["entries"]))
    print("Quality totals:", {name: int(totals[name]) for name in ["gold", "silver", "bronze"]})
else:
    print("Library summary skipped.")

## 3. Pick a working class and inspect one stored warp entry

A stored warp entry is not a light curve yet. It is a basis-SN plus original-template plus correction-surface record. The key object inside the entry is `mdict["corrmodel"]`, whose arrays represent the multiplicative correction surface `W(phase, wavelength)`.

The wavelength edge columns are usually neutral anchors near one. The interior columns are the empirically constrained broad-band correction curves.


In [ ]:
# Select a fit class and initialize the loader
preferred_fitclass = "SN II"
if available_coeffs and coeff_path_for_fitclass(preferred_fitclass).exists():
    fitclass = preferred_fitclass
elif available_coeffs:
    fitclass = fitclass_from_coeff_path(available_coeffs[0])
else:
    fitclass = preferred_fitclass

print("fitclass:", fitclass)

loader = WarpfitTemplateLoader(str(warpcoeffs_dir)) if HAS_RUNTIME_DEPS and available_coeffs else None
model_colors = loader.get_model_colors(fitclass) if loader is not None else None
print("model_colors:", model_colors)

In [ ]:
# Inspect a representative stored warp entry

def first_representative_entry(fitclass, prefer_quality="gold"):
    """Return the first usable stored warp entry, preferring a requested quality label."""
    collection = load_coeff_collection(fitclass)
    warpcoeff = collection["warpcoeff"]

    # Prefer high-quality entries that already contain the corrected model surface.
    for basis_sn, entries in warpcoeff.items():
        for entry in entries:
            if entry.get("quality") == prefer_quality and "corrmodel" in entry.get("mdict", {}):
                return basis_sn, entry, collection

    # Fall back to any entry with the required corrmodel payload.
    for basis_sn, entries in warpcoeff.items():
        for entry in entries:
            if "corrmodel" in entry.get("mdict", {}):
                return basis_sn, entry, collection

    raise RuntimeError(f"No usable corrmodel entry found for {fitclass}")


if HAS_RUNTIME_DEPS and available_coeffs:
    basis_sn, representative_entry, representative_collection = first_representative_entry(fitclass)
    corr = representative_entry["mdict"]["corrmodel"]
    phase = np.asarray(corr["phase"], dtype=float)
    wave = np.asarray(corr["wave"], dtype=float)
    flux = np.asarray(corr["flux"], dtype=float)

    print("basis_sn:", basis_sn)
    print("original template:", representative_entry.get("model"))
    print("quality:", representative_entry.get("quality"))
    print("draw_prob:", representative_entry.get("draw_prob"))
    print("peakcol:", representative_entry.get("peakcol"))
    print("phase range:", (float(phase.min()), float(phase.max())), "n=", len(phase))
    print("wave grid:", [float(value) for value in wave])
    print("flux shape:", flux.shape, "min/max:", (float(np.nanmin(flux)), float(np.nanmax(flux))))

    preview_rows = [0, len(phase) // 2, len(phase) - 1]
    preview = pd.DataFrame(
        flux[preview_rows],
        index=[f"phase={phase[index]:.1f}" for index in preview_rows],
        columns=[f"wave={value:.0f}" for value in wave],
    )
    display(preview)
else:
    print("Stored-entry inspection skipped.")

In [ ]:
# Plot original and warped templates

def preview_band_from_entry(entry, preferred="ztfr"):
    """Choose a preview band from the stored light-curve evaluation metadata."""
    lceval = entry.get("mdict", {}).get("lceval", {}) or {}
    bands = [str(band) for band in lceval.keys()]
    return preferred if preferred in bands or not bands else bands[0]


def set_preview_params(model, z=None, t0=0.0, amplitude=1.0, hostebv=0.0, hostr_v=3.1):
    """Set common preview parameters if they exist on a sncosmo model."""
    requested = {
        "z": z,
        "t0": t0,
        "amplitude": amplitude,
        "hostebv": hostebv,
        "hostr_v": hostr_v,
    }
    params = {name: value for name, value in requested.items() if value is not None and name in model.param_names}
    if params:
        model.set(**params)
    return params


def make_original_preview_model(template_name, z):
    """Build the unwarped template model with the same basic redshift setup."""
    model = sncosmo.Model(
        source=template_name,
        effects=[sncosmo.CCM89Dust()],
        effect_names=["host"],
        effect_frames=["rest"],
    )
    set_preview_params(model, z=z)
    return model


def plot_representative_warp_entry(entry, basis_sn, band=None, zp=25.0, zpsys="ab"):
    """Plot the original and warped template light curves for one representative entry."""
    template_name = entry["model"]
    z = float(entry["z"])
    band = preview_band_from_entry(entry) if band is None else band
    corr = entry["mdict"]["corrmodel"]

    original_model = make_original_preview_model(template_name, z=z)
    warped_model = get_warpedTimeSeriesModel(
        name=f"preview_{basis_sn}_{template_name}",
        original_template_name=template_name,
        warpdata=entry["mdict"],
        z=z,
        hostr_v=3.1,
        use_host_dust=True,
    )
    set_preview_params(warped_model, z=z)

    phase_grid = np.asarray(corr["phase"], dtype=float)
    observer_times = phase_grid * (1.0 + z)
    original_flux = original_model.bandflux(band, observer_times, zp=zp, zpsys=zpsys)
    warped_flux = warped_model.bandflux(band, observer_times, zp=zp, zpsys=zpsys)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(phase_grid, original_flux, color="0.45", ls="--", label=f"original template: {template_name}")
    ax.plot(phase_grid, warped_flux, color="tab:blue", lw=2, label="warped template")
    ax.axhline(0, color="0.85", lw=1)
    ax.set_ylim(0, None)
    ax.set_xlabel("Rest-frame phase [days]")
    ax.set_ylabel(f"Flux in {band} (zp={zp:g}, {zpsys})")
    ax.set_title(f"{fitclass}: {basis_sn} / {template_name}")
    ax.legend()
    ax.figure.tight_layout()

    print(f"band: {band}")
    return fig, ax


if HAS_RUNTIME_DEPS and "representative_entry" in globals():
    fig, ax = plot_representative_warp_entry(representative_entry, basis_sn)
else:
    print("Template comparison plot skipped.")

## 4. Loader selection semantics

`WarpfitTemplateLoader.get_templates()` turns stored entries into ready-to-use `sncosmo.Model` objects. It samples on two levels:

- `snbasis_selection`: which observed basis SNe to use. Use an integer for random sampling with replacement, or `"all"` for every basis SN.
- `template_selection`: how many stored template entries to select per selected basis SN. Positive integers use `draw_prob` weights; negative integers sample uniformly; `"all"` returns every valid entry for each selected basis SN.
- `min_fit_quality`: keeps entries at or above `bronze`, `silver`, or `gold` according to the loader's quality rank.

The returned list contains dictionaries with the model plus runtime provenance: `basis_sn`, `template_sn`, `template_prob`, `quality`, `peakcol`, `target_peak_color`, `samplecorr_ebv`, and `model_colors`.


In [ ]:
# Compare template-selection modes

def summarize_returned_items(items):
    """Convert loader return dictionaries into a compact inspection table."""
    rows = []

    # Loader samples are grouped by experiment label; flatten them for display.
    for label, group in items.items():
        for index, item in enumerate(group):
            model = item.get("model")
            rows.append({
                "sample": label,
                "index": index,
                "basis_sn": item.get("basis_sn"),
                "template_sn": item.get("template_sn"),
                "quality": item.get("quality"),
                "template_prob": item.get("template_prob"),
                "peakcol": item.get("peakcol"),
                "target_peak_color": item.get("target_peak_color"),
                "samplecorr_ebv": item.get("samplecorr_ebv"),
                "z": model.get("z") if model is not None and "z" in model.param_names else np.nan,
                "param_names": ", ".join(model.param_names) if model is not None else "",
            })
    return pd.DataFrame(rows)


selection_specs = {
    "weighted_two_per_basis": {
        "snbasis_selection": 3,
        "template_selection": 2,
        "min_fit_quality": "bronze",
        "random_seed": 42,
        "color_mode": None,
    },
    "uniform_two_per_basis": {
        "snbasis_selection": 3,
        "template_selection": -2,
        "min_fit_quality": "bronze",
        "random_seed": 42,
        "color_mode": None,
    },
    "all_templates_one_basis": {
        "snbasis_selection": 1,
        "template_selection": "all",
        "min_fit_quality": "bronze",
        "random_seed": 42,
        "color_mode": None,
    },
}

selection_samples = {}
if loader is not None:
    for label, kwargs in selection_specs.items():
        try:
            selection_samples[label] = loader.get_templates(fitclass=fitclass, **kwargs)
            print(label, "loaded", len(selection_samples[label]), "models")
        except Exception as exc:
            selection_samples[label] = []
            print(label, "failed:", exc)

    selection_summary = summarize_returned_items(selection_samples)
    display(selection_summary)
else:
    print("Loader sampling skipped.")

## 5. Colour modes in the current loader API

The robust baseline is `color_mode=None`: use the stored warped source as-is.

The current `loaders.py` implementation also supports three colour-hook modes:

- `"harmonize"`: use `model_colors["loc"]` as the class reference colour.
- `"draw"`: draw a target peak colour from the fitted class colour distribution stored as `K`, `loc`, and `scale`.
- `"target"`: use an explicit `target_peak_color` supplied by the caller.

All non-raw modes compare the chosen target colour to the entry's stored `peakcol`, evaluate `ebv_corr_func`, and pass the resulting `samplecorr_ebv` into source construction. This is a source-level population-colour hook, not a later observer-frame dust parameter.

A direct colour measurement is still a validation step. The chosen target colour is what the calibration attempted; interpolation details, band coverage, and the correction approximation can make the measured model colour differ slightly.


In [ ]:
# Draw samples for each colour-handling mode
color_sample_specs = {
    "raw": {"color_mode": None},
}

if model_colors is not None:
    color_sample_specs.update({
        "harmonize": {"color_mode": "harmonize"},
        "draw": {"color_mode": "draw"},
        "target_loc_plus_0p10": {
            "color_mode": "target",
            "target_peak_color": float(model_colors["loc"]) + 0.10,
        },
    })

color_samples = {}
if loader is not None:
    for label, kwargs in color_sample_specs.items():
        try:
            color_samples[label] = loader.get_templates(
                fitclass=fitclass,
                snbasis_selection=3,
                template_selection=1,
                min_fit_quality="bronze",
                random_seed=42,
                **kwargs,
            )
            print(label, "loaded", len(color_samples[label]), "models")
        except Exception as exc:
            color_samples[label] = []
            print(label, "failed:", exc)

    color_summary = summarize_returned_items(color_samples)
    display(color_summary)
else:
    print("Colour-mode sampling skipped.")

In [ ]:
# Measure peak colours in returned models

def stored_style_peak_color(model, color1, color2, magsys="ab"):
    """Measure colour from each band's source peak phase, matching stored peakcol logic."""
    phase1 = model.source.peakphase(color1)
    phase2 = model.source.peakphase(color2)
    return float(model.bandmag(color1, magsys, phase1) - model.bandmag(color2, magsys, phase2))


def colour_measurement_table(samples, max_per_mode=3):
    """Return measured colour diagnostics for up to max_per_mode samples per mode."""
    rows = []
    for label, items in samples.items():
        for item in items[:max_per_mode]:
            colors = item.get("model_colors") or model_colors
            if not colors:
                continue

            model = item["model"]
            color1 = colors["color1"]
            color2 = colors["color2"]
            row = {
                "mode": label,
                "basis_sn": item.get("basis_sn"),
                "template_sn": item.get("template_sn"),
                "stored_peakcol": item.get("peakcol"),
                "target_peak_color": item.get("target_peak_color"),
                "samplecorr_ebv": item.get("samplecorr_ebv"),
                "color_pair": f"{color1}-{color2}",
            }

            # Compare a common phase-0 colour to the colour definition stored with the library.
            try:
                row["phase0_colour"] = measure_peak_color(model, color1, color2, phase=0.0)
            except Exception as exc:
                row["phase0_colour"] = np.nan
                row["phase0_error"] = str(exc)

            try:
                row["source_peak_colour"] = stored_style_peak_color(model, color1, color2)
            except Exception as exc:
                row["source_peak_colour"] = np.nan
                row["source_peak_error"] = str(exc)

            rows.append(row)
    return pd.DataFrame(rows)


def plot_peak_colour_targets(df):
    """Visualize source peak colour, target peak colour, and their residuals."""
    if df.empty or "source_peak_colour" not in df:
        print("No peak-colour measurements available to plot.")
        return

    plot_df = df.copy().reset_index(drop=True)
    plot_df["label"] = (
        plot_df["mode"].astype(str)
        + " | "
        + plot_df["basis_sn"].astype(str)
        + " / "
        + plot_df["template_sn"].astype(str)
    )
    y = np.arange(len(plot_df))[::-1]

    source = pd.to_numeric(plot_df["source_peak_colour"], errors="coerce")
    target = pd.to_numeric(plot_df["target_peak_color"], errors="coerce")
    valid_source = source.notna()
    has_target = target.notna() & valid_source

    fig_height = max(3.0, 0.45 * len(plot_df) + 1.2)
    fig, ax = plt.subplots(figsize=(9.5, fig_height))

    ax.scatter(source[valid_source], y[valid_source], s=74, color="tab:blue", label="source_peak_colour", zorder=3)
    if has_target.any():
        delta = target - source
        for idx in plot_df.index[has_target]:
            line_color = "tab:red" if delta.loc[idx] >= 0 else "tab:purple"
            ax.plot([source.loc[idx], target.loc[idx]], [y[idx], y[idx]], color=line_color, lw=2.4, alpha=0.75, zorder=2)
            ax.scatter(target.loc[idx], y[idx], s=74, marker="D", color="tab:orange", edgecolor="black", linewidth=0.5, zorder=4)
            right_edge = max(source.loc[idx], target.loc[idx])
            ax.text(right_edge + 0.015, y[idx], f"Delta={delta.loc[idx]:+.3f}", va="center", fontsize=9)

    for idx in plot_df.index[valid_source & ~has_target]:
        ax.text(source.loc[idx] + 0.015, y[idx], "no target", va="center", fontsize=9, color="0.35")

    if model_colors is not None and "loc" in model_colors:
        ax.axvline(float(model_colors["loc"]), color="0.2", ls="--", lw=1.2, alpha=0.55, label="class loc")

    color_pair = plot_df["color_pair"].dropna().iloc[0] if plot_df["color_pair"].notna().any() else "colour"
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df["label"], fontsize=9)
    ax.set_xlabel(f"Peak colour ({color_pair}) [mag]")
    ax.set_title("Source peak colour versus target peak colour")
    ax.grid(axis="x", alpha=0.25)
    ax.scatter([], [], s=74, marker="D", color="tab:orange", edgecolor="black", linewidth=0.5, label="target_peak_color")
    ax.legend(loc="best")
    fig.tight_layout()
    plt.show()


if "color_samples" in globals() and color_samples:
    colour_checks = colour_measurement_table(color_samples)
    display(colour_checks)
    plot_peak_colour_targets(colour_checks)
else:
    print("No colour samples available to measure.")

## 6. Configure one returned `sncosmo.Model`

The loader returns parameterized models. Before turning a model into a concrete light curve, set the event-level parameters you want for this simulated object.

Keep the categories separate:

- Library/provenance: `basis_sn`, `template_sn`, `quality`, `template_prob`, `peakcol`.
- Source-level colour: `color_mode`, `target_peak_color`, `samplecorr_ebv`.
- Event-level model parameters: `z`, `t0`, `amplitude`, `hostebv`, `hostr_v`, `mwebv`, `mwr_v`.
- Observation-level choices: bands, times, zero point, magnitude system.

The helper below only sets parameters that are actually present on the model, which keeps it usable for models built with different dust options.


In [ ]:
# Configure one model for an example event

def first_available_item(*sample_groups):
    """Return the first model item found in one or more sample collections."""
    for group in sample_groups:
        if isinstance(group, dict):
            for items in group.values():
                if items:
                    return items[0]
        elif group:
            return group[0]
    return None


def set_if_present(model, **params):
    """Set only parameters that exist on a sncosmo model and return the applied values."""
    settable = {name: value for name, value in params.items() if name in model.param_names}
    if settable:
        model.set(**settable)
    return settable


selected_item = first_available_item(color_samples if "color_samples" in globals() else {}, selection_samples if "selection_samples" in globals() else {})

if selected_item is not None:
    model = selected_item["model"]
    print("Selected basis_sn:", selected_item.get("basis_sn"))
    print("Selected template_sn:", selected_item.get("template_sn"))
    print("Model parameters:", model.param_names)
    print("Current parameters before event configuration:")
    display(pd.DataFrame({"parameter": model.param_names, "value": model.parameters}))

    # Leave z at the value attached by the selected coefficient entry unless you intentionally override it.
    event_params = {
        "t0": 60000.0,
        "amplitude": 1.0,
        "hostebv": 0.0,
        "hostr_v": 3.1,
    }
    applied_event_params = set_if_present(model, **event_params)

    print("Applied event parameters:", applied_event_params)
    print("Observer-frame valid time range:", (float(model.mintime()), float(model.maxtime())))
    display(pd.DataFrame({"parameter": model.param_names, "value": model.parameters}))
else:
    print("No returned model is available yet.")

## 7. Evaluate a model as a light-curve table

`make_lightcurve_table()` is a thin package helper around `model.bandflux()`. It keeps the final step explicit: choose bands, choose observer-frame times, and evaluate the current state of the model. Bands that fail because of wavelength coverage are skipped by default and recorded in `table.meta["skipped_bands"]`.


In [ ]:
# Build a light-curve table from the selected model
if selected_item is not None:
    model = selected_item["model"]
    set_if_present(model, t0=60000.0, amplitude=1.0, hostebv=0.0, hostr_v=3.1)

    bands = ("ztfg", "ztfr", "ztfi")
    times = np.linspace(model.mintime(), model.maxtime(), 180)
    lc = make_lightcurve_table(model, bands=bands, times=times, zp=25.0, zpsys="ab")

    print("Rows:", len(lc))
    print("Skipped bands:", lc.meta.get("skipped_bands", {}))
    display(lc[:10])

    if len(lc) > 0:
        plot_lightcurve_table(lc, title=f"{fitclass}: {selected_item.get('basis_sn')} / {selected_item.get('template_sn')}")
    else:
        print("No rows were produced. Try a different band set, redshift, or template.")
else:
    print("No warped model available yet.")

In [ ]:
# Plot colour-mode light curves in one band

def plot_mode_comparison(samples, band="ztfr"):
    """Plot one representative light curve per colour mode for a chosen band."""
    fig, ax = plt.subplots(figsize=(8, 4.5))
    plotted = 0

    for label, items in samples.items():
        if not items:
            continue
        model = items[0]["model"]
        set_if_present(model, t0=60000.0, amplitude=1.0, hostebv=0.0, hostr_v=3.1)
        times = np.linspace(model.mintime(), model.maxtime(), 180)
        tab = make_lightcurve_table(model, bands=(band,), times=times, zp=25.0, zpsys="ab")
        if len(tab) == 0:
            print(label, "skipped:", tab.meta.get("skipped_bands", {}))
            continue
        ax.plot(tab["time"], tab["flux"], label=label)
        plotted += 1

    ax.axhline(0, color="0.8", lw=1)
    ax.set_xlabel("MJD / observer-frame time")
    ax.set_ylabel(f"Flux in {band}")
    ax.legend()
    ax.figure.tight_layout()
    return fig, ax, plotted


if "color_samples" in globals() and color_samples:
    fig, ax, plotted = plot_mode_comparison(color_samples, band="ztfr")
    if plotted == 0:
        print("No mode-comparison curves were plotted.")
else:
    print("No colour-mode samples available for comparison.")

## 9. Optional save

Generated light-curve tables are not written by default. Set `WRITE_OUTPUTS = True` when you want local CSV/ECSV files for downstream checks.


In [ ]:
# Optionally write generated tables to disk
WRITE_OUTPUTS = False
outdir = PACKAGE_DIR / "generated_lightcurves"

if WRITE_OUTPUTS:
    outdir.mkdir(exist_ok=True)
    if "lc" in globals():
        lc.write(outdir / "example_warped_lightcurve.ecsv", overwrite=True)
        lc.write(outdir / "example_warped_lightcurve.csv", overwrite=True)
    print("Wrote outputs to", outdir)
else:
    print("WRITE_OUTPUTS is False; no files written.")